In [0]:
%sql
use catalog cat_mini;
create schema if not exists bronze;
-- location 'dbfs:/tmp/bronze';
create schema if not exists silver;
-- location 'dbfs:/tmp/silver';
create schema if not exists gold;
-- location 'dbfs:/tmp/gold';

In [0]:
%sql
use bronze;

In [0]:
base_path ="s3://de-mini-kabhi/raw-data/"
files= dbutils.fs.ls(base_path)
csv_files=[f.path for f in files if f.name.endswith(".csv")]
display(csv_files)


In [0]:
import re
def to_snake_case(name: str) -> str:
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)
    name = re.sub(r'(?<!^)(?=[A-Z])', '_', name)
    return name.lower().strip('_')
for file in csv_files:
  table_name=file.split("/")[-1].replace("csv","").lower()
  table_name=to_snake_case(table_name)
  print(table_name)

  df=spark.read.csv(file,header=True,inferSchema=True)
  renamed_df = df.toDF(*[to_snake_case(c) for c in df.columns])
#   renamed_df.show()
  renamed_df.write.format("delta").mode("overwrite").saveAsTable(f'`{table_name}`')